In [1]:
# Section 1: Imports and Environment Setup
import cv2
import numpy as np
import csv
import os
import time
import torch
import matplotlib.pyplot as plt
import warnings

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm
from skimage.color import lab2rgb

warnings.filterwarnings("ignore", message="Failed to initialize NumPy: _ARRAY_API not found")


In [2]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
##############   TEST CELL ##########################################
import torch
checkpoint_path = '/home/mmanani/ImageColourizationDataSet/checkpoints/gan_checkpoint.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cuda'))  # Load on CPU first
    print("Checkpoint loaded successfully!")
    print("Epoch:", checkpoint['epoch'])
    print("Loss:", checkpoint['loss'])
else:
    print("Checkpoint not found!")
##############   TEST CELL ##########################################

Checkpoint not found!


In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [5]:
import os

# Base directory
base_dir = r'/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/train'

# Get all subdirectories
sub_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

# Count images in each subdirectory
for folder in sub_dirs:
    folder_path = os.path.join(base_dir, folder)
    image_count = len([f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    print(f"{folder}: {image_count} images")

# Total count
total_images = sum(len([f for f in os.listdir(os.path.join(base_dir, d)) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) 
                  for d in sub_dirs if os.path.isdir(os.path.join(base_dir, d)))
print(f"Total images: {total_images}")

ILSVRC2013_train_extra2: 9939 images
ILSVRC2013_train_extra8: 10000 images
ILSVRC2013_train_extra4: 9990 images
ILSVRC2013_train_extra10: 7505 images
ILSVRC2014_train_0001: 10000 images
ILSVRC2014_train_0002: 10000 images
ILSVRC2013_train: 0 images
ILSVRC2014_train_0000: 9999 images
ILSVRC2013_train_extra3: 9946 images
ILSVRC2013_train_extra1: 9948 images
ILSVRC2013_train_extra7: 10000 images
ILSVRC2013_train_extra9: 9999 images
ILSVRC2014_train_0004: 10000 images
ILSVRC2014_train_0005: 10000 images
ILSVRC2014_train_0003: 10000 images
ILSVRC2013_train_extra5: 9997 images
ILSVRC2013_train_extra6: 9996 images
ILSVRC2013_train_extra0: 9928 images
ILSVRC2014_train_0006: 659 images
Total images: 167906


In [18]:
# Hyperparameters
batch_size = 16
num_epochs = 100
learning_rate_g = 2e-5
learning_rate_d = 1e-5
pretrain_epochs = 10

In [8]:
# Section 2: Dataset Class (Revised)
class ColorizationDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_size=1*1024*1024*1024):
        self.image_paths = []
        total_size = 0
        for root, _, files in os.walk(image_dir):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    path = os.path.join(root, file)
                    size = os.path.getsize(path)
                    if total_size + size <= max_size:
                        self.image_paths.append(path)
                        total_size += size
                    else:
                        break
            if total_size >= max_size:
                break
        if not self.image_paths:
            raise ValueError(f"No images found in {image_dir}.")
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)    

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Failed to load {img_path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (256, 256), interpolation=cv2.INTER_AREA)  # Resize fix
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel = img_lab[:,:,0] / 255.0
        ab_channels = (img_lab[:,:,1:] - 128) / 128.0
        l_channel = torch.from_numpy(l_channel).float().unsqueeze(0)
        ab_channels = torch.from_numpy(ab_channels.transpose((2, 0, 1))).float()
        if self.transform:
            l_channel = self.transform(l_channel)
            ab_channels = self.transform(ab_channels)
        return {'L': l_channel, 'ab': ab_channels}

In [9]:
# Load dataset
image_dir = r'/home/mmanani/ImageColourizationDataSet/ILSVRC/Data/DET/train'  # Your path
dataset = ColorizationDataset(image_dir, transform=transforms.RandomHorizontalFlip())
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [10]:
# Section 3: Model Definition
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 2, 4, 2, 1), nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0), nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [11]:
# Initialize models
G = Generator().to(device)
D = Discriminator().to(device)

# Section 4: Loss Functions and Optimizers
vgg = models.vgg16(pretrained=True).features.eval().to(device)
for param in vgg.parameters():
    param.requires_grad = False

def perceptual_loss(pred, target):
    pred_features = vgg(pred)
    target_features = vgg(target)
    return nn.functional.mse_loss(pred_features, target_features)

criterion_GAN = nn.BCELoss().to(device)
criterion_L1 = nn.L1Loss().to(device)

optimizer_G = optim.Adam(G.parameters(), lr=learning_rate_g, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=learning_rate_d, betas=(0.5, 0.999))

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [12]:
# Section 5: Training Loop
# Assuming these are defined earlier: device, pretrain_epochs, num_epochs, train_loader, G, D, optimizer_G, optimizer_D, criterion_L1, perceptual_loss
# train_step definition with error fix
def train_step(G, D, optimizer_G, optimizer_D, real_ab, L):
    # Generator
    fake_ab = G(L)
    fake_img = torch.cat([L, fake_ab], dim=1)
    # Discriminator (real)
    real_img = torch.cat([L, real_ab], dim=1)
    real_pred = D(real_img)
    real_label = torch.clamp(1.0 - torch.rand_like(real_pred) * 0.1, 0.9, 1.0)
    # Discriminator (fake)
    fake_pred = D(fake_img.detach())
    fake_label = torch.clamp(torch.rand_like(fake_pred) * 0.1, 0.0, 0.1)
    # D loss
    d_loss_real = criterion_GAN(real_pred, real_label)
    d_loss_fake = criterion_GAN(fake_pred, fake_label)
    d_loss = (d_loss_real + d_loss_fake) / 2
    optimizer_D.zero_grad()
    d_loss.backward()
    optimizer_D.step()
    # G loss
    fake_pred = D(fake_img)
    g_gan_loss = criterion_GAN(fake_pred, real_label)
    g_l1_loss = criterion_L1(fake_ab, real_ab) * 100
    g_perceptual_loss = perceptual_loss(fake_img, real_img)
    g_loss = g_gan_loss + g_l1_loss + g_perceptual_loss
    optimizer_G.zero_grad()
    g_loss.backward()
    optimizer_G.step()
    return g_loss.item(), d_loss.item()
checkpoint_dir = '/home/mmanani/ImageColourizationDataSet/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Function to load checkpoint (updated for GAN to load both G and D)
def load_checkpoint(checkpoint_path, models, optimizers):
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        for key in models:
            models[key].load_state_dict(checkpoint['model_state_dict'][key])
        for key in optimizers:
            optimizers[key].load_state_dict(checkpoint['optimizer_state_dict'][key])
        start_epoch = checkpoint['epoch'] + 1
        start_batch = checkpoint['batch'] if 'batch' in checkpoint else 0
        loss = checkpoint['loss']
        print(f"Resumed from checkpoint at epoch {checkpoint['epoch']}, batch {start_batch}, loss {loss}")
        return start_epoch, start_batch
    return 0, 0

# -----------------------------
# 2. PER-EPOCH COPY (MAIN CHECKPOINT UNCHANGED)
# -----------------------------
def save_per_epoch_copy(epoch, checkpoint_dir="checkpoints"):
    main_path = os.path.join(checkpoint_dir, 'gan_checkpoint.pth')
    if not os.path.exists(main_path):
        return
    copy_path = os.path.join(checkpoint_dir, f'gan_checkpoint_epoch_{epoch+1}.pth')
    import shutil
    shutil.copy2(main_path, copy_path)
    print(f"Epoch copy saved: {copy_path}")


In [13]:

# Pretraining Generator (assuming train_loader is defined with shuffle=False for resumption)
print("Pretraining Generator...")
start_epoch, start_batch = load_checkpoint(os.path.join(checkpoint_dir, 'pretrain_checkpoint.pth'), G, optimizer_G)
for epoch in range(start_epoch, pretrain_epochs):
    total_g_loss = 0
    start_time = time.time()
    if epoch == start_epoch and start_batch > 0:
        iterator = iter(train_loader)
        for _ in range(start_batch):  # Skip to the resume batch
            next(iterator)
        batches = iterator
    else:
        batches = train_loader

    for batch_idx, batch in enumerate(tqdm(batches, desc=f"Pretrain Epoch {epoch+1}/{pretrain_epochs}", total=len(train_loader))):
        if epoch == start_epoch and batch_idx < start_batch:
            continue
        L = batch['L'].to(device)
        real_ab = batch['ab'].to(device)
        fake_ab = G(L)
        g_l1_loss = criterion_L1(fake_ab, real_ab) * 100
        g_perceptual_loss = perceptual_loss(torch.cat([L, fake_ab], dim=1), torch.cat([L, real_ab], dim=1))
        g_loss = g_l1_loss + g_perceptual_loss
        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()
        total_g_loss += g_loss.item()
        current_time = time.time()
        elapsed_time = current_time - start_time
        if elapsed_time >= 600:
            #print(f"Pausing for 1 minute at {elapsed_time:.1f} seconds to cool down GPU...")
            time.sleep(60)
            start_time = current_time
    avg_g_loss = total_g_loss / len(train_loader)
    print(f"Pretrain Epoch [{epoch+1}/{pretrain_epochs}], G Loss: {avg_g_loss:.4f}")
    torch.save({
        'epoch': epoch+1,
        'batch': 0,
        'model_state_dict': G.state_dict(),
        'optimizer_state_dict': optimizer_G.state_dict(),
        'loss': avg_g_loss
    }, os.path.join(checkpoint_dir, 'pretrain_checkpoint.pth'))

Pretraining Generator...


Pretrain Epoch 1/10: 100%|██████████| 593/593 [04:00<00:00,  2.47it/s]


Pretrain Epoch [1/10], G Loss: 61.4285


Pretrain Epoch 2/10: 100%|██████████| 593/593 [03:53<00:00,  2.54it/s]


Pretrain Epoch [2/10], G Loss: 59.9240


...

Pretrain Epoch [8/10], G Loss: 59.4624


Pretrain Epoch 9/10: 100%|██████████| 593/593 [03:50<00:00,  2.58it/s]


Pretrain Epoch [9/10], G Loss: 59.4843


Pretrain Epoch 10/10: 100%|██████████| 593/593 [03:57<00:00,  2.50it/s]

Pretrain Epoch [10/10], G Loss: 59.4120


In [14]:
# Section 6: Inference and Visualization (Optional)
def colorize_image(image_path, G, device='cuda'):
    G.eval()
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    l_channel = lab[:, :, 0]
    # 01/11/2025
    # ✅ CHANGED: match normalization to dataset (0–255 scaled)
    # l_resized = cv2.resize(l_channel, (256, 256)) / 100.0   # old
    l_resized = cv2.resize(l_channel, (256, 256)) / 255.0
    l_input = torch.from_numpy(l_resized).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        ab_pred = G(l_input)
        ab_pred = torch.clamp(ab_pred, -1, 1)
        ab_pred = ab_pred.squeeze(0).cpu().numpy() * 128
    ab_resized = cv2.resize(ab_pred.transpose(1,2,0), (w, h), cv2.INTER_CUBIC)
    #01/11/2025 ✅ NEW: add Gaussian smoothing for smoother chroma transitions
    ab_resized = cv2.GaussianBlur(ab_resized, (5, 5), 0)
    np.clip(ab_resized, -128, 127, out=ab_resized)
    l_original = cv2.resize(l_channel, (w, h))
    lab_full = np.dstack((l_original, ab_resized)).astype(np.float64)
    rgb = lab2rgb(lab_full)
    return (rgb * 255).astype(np.uint8)

# ================================
# 1. SMOOTHNESS LOSS FUNCTION
# ================================
def smoothness_loss(ab):
    """Encourage spatial smoothness in ab channels"""
    dx = torch.abs(ab[:, :, :, 1:] - ab[:, :, :, :-1])
    dy = torch.abs(ab[:, :, 1:, :] - ab[:, :, :-1, :])
    return (dx.mean() + dy.mean()) * 0.1 #0.5-> 0.1


In [ ]:
print(learning_rate_g)
print(learning_rate_d)
# GAN Training
# -----------------------------
# 3. CSV INIT
# -----------------------------
csv_file = 'training_metrics.csv'
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Batch', 'G_Loss', 'D_Loss', 'PSNR', 'SSIM'])
print("Starting GAN Training...")

# -----------------------------
# 4. LOAD CHECKPOINT (TERA ORIGINAL)
# -----------------------------
models = {'G': G, 'D': D}
optimizers = {'G': optimizer_G, 'D': optimizer_D}
start_epoch, start_batch = load_checkpoint(os.path.join(checkpoint_dir, 'gan_checkpoint.pth'), models, optimizers)

if start_epoch > 0:
    start_epoch += 1
    print(f"Resuming training from Epoch {start_epoch}")

# Load pretrain weights if fresh
if start_epoch == 0:
    pretrain_path = os.path.join(checkpoint_dir, 'pretrain_checkpoint.pth')
    if os.path.exists(pretrain_path):
        G.load_state_dict(torch.load(pretrain_path)['model_state_dict'])
        print("Loaded pretrain weights for G")

# -----------------------------
# 5. TRAINING LOOP (ALL FIXES + BUG FIXED)
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    G.train()
    D.train()
    total_g_loss = total_d_loss = 0.0
    metrics_list = []
    start_time = time.time()

    # Resume mid-epoch
    if epoch == start_epoch and start_batch > 0:
        iterator = iter(train_loader)
        for _ in range(start_batch):
            next(iterator)
        batches = iterator
    else:
        batches = train_loader

    # Progress bar
    pbar = tqdm(batches, desc=f"GAN Epoch {epoch}/{num_epochs}", total=len(train_loader))

    for batch_idx, batch in enumerate(pbar):
        if epoch == start_epoch and batch_idx < start_batch:
            continue

        L = batch['L'].to(device)
        real_ab = batch['ab'].to(device)

        # -----------------------------
        # DISCRIMINATOR (ALREADY FIXED)
        # -----------------------------
        optimizer_D.zero_grad()
        real_input = torch.cat([L, real_ab], dim=1)
        fake_ab_detached = G(L).detach()
        fake_input = torch.cat([L, fake_ab_detached], dim=1)

        real_pred = D(real_input)
        fake_pred = D(fake_input)

        d_loss_real = F.binary_cross_entropy_with_logits(real_pred, torch.ones_like(real_pred))
        d_loss_fake = F.binary_cross_entropy_with_logits(fake_pred, torch.zeros_like(fake_pred))
        d_loss = (d_loss_real + d_loss_fake) * 0.5
        d_loss.backward()
        optimizer_D.step()

        # -----------------------------
        # GENERATOR (NOW 100% FIXED)
        # -----------------------------
        optimizer_G.zero_grad()
        fake_ab = G(L)
        fake_ab_clamped = torch.clamp(fake_ab, -1.0, 1.0) #0.9 -> 1.0
        fake_input = torch.cat([L, fake_ab_clamped], dim=1)   # ← YEH DAAL

        g_gan = F.binary_cross_entropy_with_logits(D(fake_input), torch.ones_like(real_pred))
        g_l1 = F.l1_loss(fake_ab_clamped, real_ab) * 80 # 150 -> 80
        g_smooth = smoothness_loss(fake_ab_clamped)
        g_loss = g_gan + g_l1 + g_smooth

        g_loss.backward()
        optimizer_G.step()


        # -----------------------------
        # 3. METRICS
        # -----------------------------
        with torch.no_grad():
            psnr_score = ssim_score = None
            if real_ab is not None:
                fake_rgb = torch.cat([L, fake_ab_clamped], dim=1).cpu().numpy()[0].transpose(1,2,0) * 255
                real_rgb = torch.cat([L, real_ab], dim=1).cpu().numpy()[0].transpose(1,2,0) * 255
                psnr_score = psnr(real_rgb, fake_rgb, data_range=255)
                if real_rgb.shape[0] >= 3 and real_rgb.shape[1] >= 3:
                    ssim_score = ssim(real_rgb, fake_rgb, multichannel=True, data_range=255, win_size=3)

            g_val = g_loss.item()
            d_val = d_loss.item()
            metrics_list.append([epoch, batch_idx, g_val, d_val, psnr_score, ssim_score])
            total_g_loss += g_val
            total_d_loss += d_val

            pbar.set_postfix({'G': f'{g_val:.4f}', 'D': f'{d_val:.4f}'})

    # -----------------------------
    # EPOCH END: CSV + TEST + CHECKPOINT
    # -----------------------------
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(metrics_list)

    avg_g = total_g_loss / len(train_loader)
    avg_d = total_d_loss / len(train_loader)
    print(f"GAN Epoch [{epoch}/{num_epochs}], G Loss: {avg_g:.4f}, D Loss: {avg_d:.4f}")

    # Test images
    test_img1 = '/home/mmanani/ImageColourizationDataSet/sourceimages/Wild-Horse-Wyoming-3284-Edit-Edit-BW.jpg'
    test_img2 = '/home/mmanani/ImageColourizationDataSet/sourceimages/Tips-Black-and-White-Portrait-Photography.jpg'

    G.eval()
    with torch.no_grad():
        colorized1 = colorize_image(test_img1, G)
        colorized2 = colorize_image(test_img2, G)
    plt.imsave(f"/home/mmanani/ImageColourizationDataSet/generatedcolouredimages/test_epoch_{epoch+1}.jpg", colorized1)
    plt.imsave(f"/home/mmanani/ImageColourizationDataSet/generatedcolouredimages/test2_epoch_{epoch+1}.jpg", colorized2)

    # MAIN CHECKPOINT (UNCHANGED)
    torch.save({
        'epoch': epoch,
        'batch': 0,
        'model_state_dict': {'G': G.state_dict(), 'D': D.state_dict()},
        'optimizer_state_dict': {'G': optimizer_G.state_dict(), 'D': optimizer_D.state_dict()},
        'loss': {'G': avg_g, 'D': avg_d}
    }, os.path.join(checkpoint_dir, 'gan_checkpoint.pth'))

    # EXTRA COPY PER EPOCH
    save_per_epoch_copy(epoch, checkpoint_dir)

# -----------------------------
# FINAL SAVE
# -----------------------------
torch.save(G.state_dict(), '/home/mmanani/ImageColourizationDataSet/generator_final.pth')
torch.save(D.state_dict(), '/home/mmanani/ImageColourizationDataSet/discriminator_final.pth')
print("GAN Training completed!")

2e-05
1e-05
Starting GAN Training...
Resumed from checkpoint at epoch 42, batch 0, loss {'G': 47.40111694528924, 'D': 0.5032050453792334}
Resuming training from Epoch 44


GAN Epoch 44/100: 100%|██████████| 593/593 [02:03<00:00,  4.81it/s, G=42.9535, D=0.5032]


GAN Epoch [44/100], G Loss: 47.3876, D Loss: 0.5032


/tmp/ipykernel_6057/125549605.py:24: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 32 negative Z values that have been clipped to zero
  rgb = lab2rgb(lab_full)


Epoch copy saved: /home/mmanani/ImageColourizationDataSet/checkpoints/gan_checkpoint_epoch_45.pth


...

Epoch copy saved: /home/mmanani/ImageColourizationDataSet/checkpoints/gan_checkpoint_epoch_99.pth


GAN Epoch 99/100: 100%|██████████| 593/593 [02:05<00:00,  4.73it/s, G=41.3721, D=0.5032]


GAN Epoch [99/100], G Loss: 47.1979, D Loss: 0.5032


/tmp/ipykernel_6057/125549605.py:24: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 77 negative Z values that have been clipped to zero
  rgb = lab2rgb(lab_full)


Epoch copy saved: /home/mmanani/ImageColourizationDataSet/checkpoints/gan_checkpoint_epoch_100.pth
GAN Training completed!


In [ ]:
# Visualize
sample_path = '/home/mmanani/ImageColourizationDataSet/sourceimages/Wild-Horse-Wyoming-3284-Edit-Edit-BW.jpg'
sample_path2 = '/home/mmanani/ImageColourizationDataSet/sourceimages/Tips-Black-and-White-Portrait-Photography.jpg'
colorized_img = colorize_image(sample_path, G)
colorized_img2 = colorize_image(sample_path2, G)
plt.imshow(colorized_img)
plt.imshow(colorized_img2)
plt.title("Colorized Image")
plt.axis('off')
plt.show()



In [ ]:
#######################################################################
############# JUST A BACKUP NOT FOR RUNNIG ############################
#######################################################################

# Initialize CSV file (Cell 4 ya 6 ke shuru mein)
csv_file = 'training_metrics.csv'
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Batch', 'G_Loss', 'D_Loss', 'PSNR', 'SSIM'])        
print("Starting GAN Training...")
models = {'G': G, 'D': D}
optimizers = {'G': optimizer_G, 'D': optimizer_D}
start_epoch, start_batch = load_checkpoint(os.path.join(checkpoint_dir, 'gan_checkpoint.pth'), models, optimizers)
if start_epoch > 0:
    start_epoch += 1   # ← YEHI MISSING THI!
    print(f"Resuming training from Epoch {start_epoch}")# Load pretrain weights for G if GAN fresh
if start_epoch == 0:
    g_checkpoint = torch.load(os.path.join(checkpoint_dir, 'pretrain_checkpoint.pth'))
    G.load_state_dict(g_checkpoint['model_state_dict'])
    print("Loaded pretrain weights for G")
    for epoch in range(start_epoch, num_epochs):
    total_g_loss, total_d_loss = 0, 0
    metrics_list = []  # Store metrics per epoch
    start_time = time.time()
    if epoch == start_epoch and start_batch > 0:
        iterator = iter(train_loader)
        for _ in range(start_batch):
            next(iterator)
        batches = iterator
    else:
        batches = train_loader
    for batch_idx, batch in enumerate(tqdm(batches, desc=f"GAN Epoch {epoch+1}/{num_epochs}", total=len(train_loader))):
    if epoch == start_epoch and batch_idx < start_batch:
        continue
    L = batch['L'].to(device)
    real_ab = batch['ab'].to(device)
    L = L.detach()
    real_ab = real_ab.detach()
    g_loss, d_loss = train_step(G, D, optimizer_G, optimizer_D, real_ab, L)
    # Metrics calculation (if ground truth available)
    psnr_score = None
    ssim_score = None
    if real_ab is not None:
        fake_ab = G(L)
        fake_ab = fake_ab.detach()  # ← ADD YE LINE
        fake_rgb = torch.cat((L, fake_ab), dim=1).detach()
        real_rgb = torch.cat((L, real_ab), dim=1).detach()
        fake_rgb = fake_rgb.cpu().numpy().transpose(0, 2, 3, 1) * 255
        real_rgb = real_rgb.cpu().numpy().transpose(0, 2, 3, 1) * 255
        psnr_score = psnr(real_rgb[0], fake_rgb[0], data_range=255)
        # Replace SSIM line
        if real_rgb.shape[0] >= 3 and real_rgb.shape[1] >= 3:
            ssim_score = ssim(real_rgb[0], fake_rgb[0], multichannel=True, data_range=255, win_size=3)
        else:
            ssim_score = 0.0
        # Check if g_loss and d_loss are tensors before .item()
        g_loss_val = g_loss.item() if torch.is_tensor(g_loss) else g_loss
        d_loss_val = d_loss.item() if torch.is_tensor(d_loss) else d_loss
        metrics_list.append([epoch, batch_idx, g_loss_val, d_loss_val, psnr_score, ssim_score])
    
    total_g_loss += g_loss.item() if torch.is_tensor(g_loss) else g_loss
    total_d_loss += d_loss.item() if torch.is_tensor(d_loss) else d_loss
    #elapsed_time = time.time() - start_time
    #if elapsed_time >= 600:
        #print(f"Pausing for 1 min at {elapsed_time:.1f}s...")
        #time.sleep(60)
        #start_time = time.time()

# Save metrics to CSV at epoch end
with open(csv_file, 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(metrics_list)

avg_g = total_g_loss / len(train_loader)
avg_d = total_d_loss / len(train_loader)
# Training loop mein har 5 epoch pe test kar
test_img = '/home/mmanani/ImageColourizationDataSet/sourceimages/Wild-Horse-Wyoming-3284-Edit-Edit-BW.jpg'
test_img2 = '/home/mmanani/ImageColourizationDataSet/sourceimages/Tips-Black-and-White-Portrait-Photography.jpg'

colorized = colorize_image(test_img, G)  # Upar wala inference function
colorized2 = colorize_image(test_img2, G)  # Upar wala inference function

plt.imsave(f"/home/mmanani/ImageColourizationDataSet/generatedcolouredimages/test_epoch_{epoch+1}.jpg", colorized)
plt.imsave(f"/home/mmanani/ImageColourizationDataSet/generatedcolouredimages/test2_epoch_{epoch+1}.jpg", colorized2)
#print(f"Test saved: test_epoch_{epoch+1}.jpg")
print(f"GAN Epoch [{epoch+1}/{num_epochs}], G Loss: {avg_g:.4f}, D Loss: {avg_d:.4f}")
G.eval()  # ← Inference ke liye zaroori!
torch.save({
    'epoch': epoch+1,
    'batch': 0,
    'model_state_dict': {'G': G.state_dict(), 'D': D.state_dict()},
    'optimizer_state_dict': {'G': optimizer_G.state_dict(), 'D': optimizer_D.state_dict()},
    'loss': {'G': avg_g, 'D': avg_d}
}, os.path.join(checkpoint_dir, 'gan_checkpoint.pth'))
save_per_epoch_copy(epoch, checkpoint_dir)torch.save(G.state_dict(), '/home/mmanani/ImageColourizationDataSet/generator_final.pth')
torch.save(D.state_dict(), '/home/mmanani/ImageColourizationDataSet/discriminator_final.pth')
print("GAN Training completed!")